# 03a — AI labelling of the Mentions/Reasons lexicon


In [ ]:
import os, re, time, datetime
import pandas as pd
import requests
from getpass import getpass
from concurrent.futures import ThreadPoolExecutor

MODEL = "gpt-5.4-nano"                 
PROMPT_FILE = "ai_labels/chatgpt_prompt.txt"     
PROMPT = open(PROMPT_FILE, encoding="utf-8").read()
API_URL = "https://api.openai.com/v1/chat/completions"

sample = pd.read_csv("ai_labels/check_sample.csv", encoding="utf-8-sig", keep_default_na=False)   
print(sample.shape, list(sample.columns))      
print("Prompt loaded from", PROMPT_FILE, ":", len(PROMPT.split()), "words")

(400, 2) ['review_id', 'text_clean']
Prompt loaded from chatgpt_prompt.txt : 225 words


In [ ]:
labels_done = os.path.exists("ai_labels/chatgpt_api_labels.csv")
if labels_done:
    print("chatgpt_api_labels.csv already exists - no API key needed, no tokens used.")
else:
    os.environ["OPENAI_API_KEY"] = getpass("Paste your OpenAI API key: ")   
HEADERS = {"Authorization": f"Bearer {os.environ.get('OPENAI_API_KEY', '')}",
           "Content-Type": "application/json"}

def label_review(review_id, text):
    """Send ONE review with the prompt. Returns (mentions, reasons, model used, raw answer)."""
    user_msg = (f"Label this one review.\nreview_id: {review_id}\ntext: {text}\n\n"
                "Return only one CSV line: review_id,gpt_mentions,gpt_reasons")
    body = {"model": MODEL, "temperature": 0,
            "messages": [{"role": "system", "content": PROMPT},
                         {"role": "user", "content": user_msg}]}
    for attempt in range(4):
        try:
            r = requests.post(API_URL, headers=HEADERS, json=body, timeout=60)
        except requests.RequestException as e:
            print("Network problem:", e); time.sleep(5); continue
        if r.status_code == 200:
            data = r.json()
            raw = data["choices"][0]["message"]["content"].strip()
            answers = re.findall(r"\b(YES|NO)\b", raw.upper())  
            if len(answers) >= 2:
                return answers[-2], answers[-1], data["model"], raw
            return "", "", data["model"], raw                   
        print("API error", r.status_code, ":", r.text[:300])
        if r.status_code in (401, 403, 404):
            break
        time.sleep(5)
    return "", "", "", ""


if labels_done:
    print("Test skipped - labels already exist (no API calls, no tokens used).")
else:
    for _, row in sample.head(3).iterrows():
        m, rsn, used, raw = label_review(row["review_id"], row["text_clean"])
        print(row["review_id"], "| mentions:", m, "| reasons:", rsn, "| raw:", raw[:60])
    print("Model used:", used)

chatgpt_api_labels.csv already exists - no API key needed, no tokens used.
Test skipped - labels already exist (no API calls, no tokens used).


In [ ]:
OUT = "ai_labels/chatgpt_api_labels.csv"
if os.path.exists(OUT):
    print(OUT, "already exists - not overwritten.")
else:
    def work(row):
        return (row["review_id"],) + label_review(row["review_id"], row["text_clean"])

    with ThreadPoolExecutor(max_workers=8) as pool:        
        results = list(pool.map(work, [r for _, r in sample.iterrows()]))

    labels = pd.DataFrame(results, columns=["review_id", "gpt_mentions", "gpt_reasons",
                                            "model_used", "raw_answer"])
    labels["date"] = str(datetime.date.today())
    labels.to_csv(OUT, index=False, encoding="utf-8-sig")

    print("Saved", OUT, labels.shape)
    print("Mentions:", labels["gpt_mentions"].value_counts().to_dict())
    print("Reasons:", labels["gpt_reasons"].value_counts().to_dict())
    print("Missing answers (should be 0):", (labels["gpt_mentions"] == "").sum())
    print("Model(s):", labels["model_used"].unique())

chatgpt_api_labels.csv already exists - not overwritten. Delete it first if you really want to redo it.


In [ ]:
for f in ["ai_labels/check_sample_answers.csv", "manual_checks/spot_check.csv"]:
    print(f, list(pd.read_csv(f, encoding="utf-8-sig").columns))

check_sample_answers.csv ['review_id', 'list_mentions', 'list_reasons']
spot_check.csv ['review_id', 'text_clean', 'my_mentions', 'my_reasons']


In [ ]:
from sklearn.metrics import cohen_kappa_score

def yn(series):
    """Turn True/False, 1/0, yes/no into 'YES'/'NO'."""
    return series.astype(str).str.strip().str.upper().replace(
        {"TRUE": "YES", "FALSE": "NO", "1": "YES", "0": "NO", "Y": "YES", "N": "NO"})

gpt = pd.read_csv("ai_labels/chatgpt_api_labels.csv", encoding="utf-8-sig", keep_default_na=False)
answers = pd.read_csv("ai_labels/check_sample_answers.csv", encoding="utf-8-sig", keep_default_na=False)
mine = pd.read_csv("manual_checks/spot_check.csv", encoding="utf-8-sig", keep_default_na=False)

m = gpt.merge(mine, on="review_id")
print("Reviews you hand-labelled:", len(m))
for name, g_col, my_col in [("Mentions", "gpt_mentions", "my_mentions"),
                            ("Reasons", "gpt_reasons", "my_reasons")]:
    g, y = yn(m[g_col]), yn(m[my_col])
    print(f"{name}: agreement {(g == y).mean() * 100:.1f}% | "
          f"kappa {cohen_kappa_score(g, y):.2f}  (0.61+ = substantial)")

a = gpt.merge(answers, on="review_id")
for name, g_col, l_col in [("Mentions", "gpt_mentions", "list_mentions"),
                           ("Reasons", "gpt_reasons", "list_reasons")]:
    g, lst = yn(a[g_col]), yn(a[l_col])
    precision = (g[lst == "YES"] == "YES").mean()    
    miss_rate = (g[lst == "NO"] == "YES").mean()     
    n_yes, n_no = (lst == "YES").sum(), (lst == "NO").sum()
    print(f"\n{name}: list YES = {n_yes}, list NO = {n_no}")
    print(f"  Precision (AI agrees when list says YES): {precision * 100:.1f}%")
    print(f"  Miss rate (AI says YES when list says NO): {miss_rate * 100:.1f}%")

Reviews you hand-labelled: 30
Mentions: agreement 83.3% | kappa 0.66  (0.61+ = substantial)
Reasons: agreement 66.7% | kappa 0.38  (0.61+ = substantial)

Mentions: list YES = 200, list NO = 200
  Precision (AI agrees when list says YES): 88.5%
  Miss rate (AI says YES when list says NO): 8.5%

Reasons: list YES = 200, list NO = 200
  Precision (AI agrees when list says YES): 57.5%
  Miss rate (AI says YES when list says NO): 38.0%
